# Environment Setup — Fintech Lakeflow Jobs

**Objective:** Prepare the Databricks environment for a fintech credit orchestration project by creating a dedicated Unity Catalog schema and three Delta tables containing synthetic customer, credit application, and installment data.

## Why?

**For our project:** Provides the source tables required to implement and test independent tasks, dependencies, parameterization, and workflow orchestration using Lakeflow Jobs.

**For Data Engineering:** Demonstrates environment preparation, data organization with Unity Catalog, Delta table creation, and the separation of setup operations from orchestrated data processing.

#### 01 — Create Project Schema

Create the `fintech_jobs` schema in the `workspace` catalog to organize the project's Delta tables.

Verify that the schema exists by listing the available schemas in Unity Catalog.

In [ ]:
-- Create the schema for the Fintech Lakeflow orchestration project if it does not already exist.
CREATE SCHEMA IF NOT EXISTS workspace.fintech_jobs;

-- List all schemas in the workspace catalog to verify that fintech_jobs was created.
SHOW SCHEMAS IN workspace;

databaseName
default
fintech_jobs
information_schema


#### 02 — Create Credit Customers Source Table

Create the `credit_customers` Delta table and insert five synthetic customer records.

Each customer has a unique identifier, name, state, and registration date.

Validate the inserted records using a SQL query.

In [ ]:
-- Create the source table for fintech credit customers.
CREATE OR REPLACE TABLE workspace.fintech_jobs.credit_customers (
    customer_id BIGINT,
    customer_name STRING,
    state STRING,
    registration_date DATE
)
USING DELTA;

-- Insert five synthetic customer records.
INSERT INTO workspace.fintech_jobs.credit_customers
VALUES
    (101, 'Ana Ribeiro', 'PE', DATE '2026-10-01'),
    (102, 'Carlos Lima', 'SP', DATE '2026-10-02'),
    (103, 'Maria Costa', 'RJ', DATE '2026-10-03'),
    (104, 'Pedro Alves', 'MG', DATE '2026-10-04'),
    (105, 'Julia Santos', 'BA', DATE '2026-10-05');

-- Verify the inserted records.
SELECT *
FROM workspace.fintech_jobs.credit_customers
ORDER BY customer_id;

customer_id,customer_name,state,registration_date
101,Ana Ribeiro,PE,2026-10-01
102,Carlos Lima,SP,2026-10-02
103,Maria Costa,RJ,2026-10-03
104,Pedro Alves,MG,2026-10-04
105,Julia Santos,BA,2026-10-05


#### 03 — Create Credit Applications Source Table

Create the `credit_applications` Delta table and insert six synthetic credit applications associated with existing customers.

Each application contains a unique identifier, customer reference, application date, requested amount, and approval status.

The data includes approved, pending, and rejected applications.

Validate the inserted records using a SQL query.

In [ ]:
-- Create the source table for fintech credit applications.
CREATE OR REPLACE TABLE workspace.fintech_jobs.credit_applications (
    application_id BIGINT,
    customer_id BIGINT,
    application_date DATE,
    requested_amount DECIMAL(12,2),
    status STRING
)
USING DELTA;

-- Insert six synthetic credit applications.
INSERT INTO workspace.fintech_jobs.credit_applications
VALUES
    (2001, 101, DATE '2026-10-06', 3500.00, 'APPROVED'),
    (2002, 102, DATE '2026-10-07', 8200.00, 'PENDING'),
    (2003, 103, DATE '2026-10-08', 5000.00, 'REJECTED'),
    (2004, 104, DATE '2026-10-09', 12000.00, 'APPROVED'),
    (2005, 105, DATE '2026-10-10', 2600.00, 'APPROVED'),
    (2006, 101, DATE '2026-10-11', 4500.00, 'PENDING');

-- Verify the inserted applications.
SELECT *
FROM workspace.fintech_jobs.credit_applications
ORDER BY application_id;

application_id,customer_id,application_date,requested_amount,status
2001,101,2026-10-06,3500.00,APPROVED
2002,102,2026-10-07,8200.00,PENDING
2003,103,2026-10-08,5000.00,REJECTED
2004,104,2026-10-09,12000.00,APPROVED
2005,105,2026-10-10,2600.00,APPROVED
2006,101,2026-10-11,4500.00,PENDING


#### 04 — Create Credit Installments Source Table

Create the `credit_installments` Delta table and insert six synthetic installment records associated with approved credit applications.

Each installment contains an identifier, application reference, customer reference, due date, amount due, and payment status.

The dataset includes paid and pending installments.

Validate the inserted records using a SQL query.

In [ ]:
-- Create the source table for fintech credit installments.
CREATE OR REPLACE TABLE workspace.fintech_jobs.credit_installments (
    installment_id BIGINT,
    application_id BIGINT,
    customer_id BIGINT,
    due_date DATE,
    amount_due DECIMAL(12,2),
    payment_status STRING
)
USING DELTA;

-- Insert six synthetic installments for approved credit applications.
INSERT INTO workspace.fintech_jobs.credit_installments
VALUES
    (3001, 2001, 101, DATE '2026-10-20', 1750.00, 'PAID'),
    (3002, 2001, 101, DATE '2026-11-20', 1750.00, 'PENDING'),
    (3003, 2004, 104, DATE '2026-10-25', 4000.00, 'PAID'),
    (3004, 2004, 104, DATE '2026-11-25', 4000.00, 'PENDING'),
    (3005, 2004, 104, DATE '2026-12-25', 4000.00, 'PENDING'),
    (3006, 2005, 105, DATE '2026-11-10', 2600.00, 'PENDING');

-- Verify the inserted installments.
SELECT *
FROM workspace.fintech_jobs.credit_installments
ORDER BY installment_id;

installment_id,application_id,customer_id,due_date,amount_due,payment_status
3001,2001,101,2026-10-20,1750.00,PAID
3002,2001,101,2026-11-20,1750.00,PENDING
3003,2004,104,2026-10-25,4000.00,PAID
3004,2004,104,2026-11-25,4000.00,PENDING
3005,2004,104,2026-12-25,4000.00,PENDING
3006,2005,105,2026-11-10,2600.00,PENDING


#### Create the File Arrival Volume

**Objective:** Create a volume to store installment CSV files.

**Why?** This volume provides the input location for the ingestion notebook and the path that the file arrival trigger will monitor.

In [ ]:
%sql
-- Create the managed volume for installment CSV files.
CREATE VOLUME IF NOT EXISTS
    workspace.fintech_jobs.trigger_storage_location;

-- Verify that the volume exists in the project schema.
SHOW VOLUMES IN workspace.fintech_jobs;

database,volume_name
fintech_jobs,trigger_storage_location


#### Prepare the Credit Installments CSV

**Objective:** Export the existing synthetic installment records to a CSV in the project volume.

**Why?** Provides a reproducible file input for testing the ingestion notebook.

In [ ]:
%python
# Define the destination file inside the volume.
volume_path = "/Volumes/workspace/fintech_jobs/trigger_storage_location/"
csv_path = f"{volume_path}credit_installments.csv"

# Export the small dataset as one CSV with headers.
installments_pdf = (
    spark.table("workspace.fintech_jobs.credit_installments").toPandas()
)
installments_pdf.to_csv(csv_path, index=False)

# Read the exported CSV to verify access and record count.
csv_df = spark.read.option("header", "true").csv(csv_path)

# Display the results for validation.
print(f"Source records: {len(installments_pdf)}")
print(f"CSV records: {csv_df.count()}")
print(f"CSV path: {csv_path}")

Source records: 6
CSV records: 6
CSV path: /Volumes/workspace/fintech_jobs/trigger_storage_location/credit_installments.csv


#### Validate the File Arrival Trigger

**Objective:** Automatically execute `fintech_credit_scheduling_job`
when a new file arrives in the monitored volume.

**Data Engineering value:** Connect file delivery to workflow execution,
reducing manual intervention and making ingestion observable.

**Expected result:** A Job run launched “By file arrival,”
with all three tasks showing “Succeeded.”

#### Simulate a New File Arrival

**Objective:** Test automatic Job execution by giving the existing
installment CSV a unique filename.

**Why:** Validate file arrival detection while preserving one input
file and the original six records.

In [ ]:
%python
from uuid import uuid4

# Define the existing file and a unique destination filename.
volume_path = "/Volumes/workspace/fintech_jobs/trigger_storage_location/"
source_path = f"{volume_path}credit_installments.csv"
arrival_path = f"{volume_path}credit_installments_{uuid4().hex}.csv"

# Rename the test file without retaining a second copy.
moved = dbutils.fs.mv(source_path, arrival_path)

# Verify the operation and the volume contents.
print(f"File renamed: {moved}")
print(f"New file: {arrival_path}")
display(dbutils.fs.ls(volume_path))

File renamed: True
New file: /Volumes/workspace/fintech_jobs/trigger_storage_location/credit_installments_021c0ecfc6a04046b2347f35396af91b.csv


path,name,size,modificationTime
dbfs:/Volumes/workspace/fintech_jobs/trigger_storage_location/credit_installments_021c0ecfc6a04046b2347f35396af91b.csv,credit_installments_021c0ecfc6a04046b2347f35396af91b.csv,317,1793468659000
